# AI Web Search (v1)

A minimal RAG-based AI web search demo.

**Workflow:**
1. User submits a query
2. **Web Search** — Tavily fetches the top 10 results (title, URL, snippet)
3. **LLM Generation** — a single LiteLLM call produces two sections:
   - **Answer**: synthesized response with inline citations linking to source URLs
   - **Web Search Results**: all 10 results listed with clickable titles and snippets
4. **Display** — markdown output is converted to HTML and rendered in Colab

In [ ]:
# Install required packages
!pip install tavily-python litellm markdown --quiet

## Configuration

Set your API keys and choose a model before running.

- **TAVILY_API_KEY** — get one at [tavily.com](https://tavily.com)
- **OPENAI_API_KEY** — get one at [platform.openai.com](https://platform.openai.com)
- **MODEL** — any LiteLLM-supported model string; defaults to `gpt-4o-mini` (fast and cheap)

In [ ]:
import os

TAVILY_API_KEY = "tvly-..."   # get from tavily.com
OPENAI_API_KEY = "sk-..."     # get from platform.openai.com

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY  # make key available to LiteLLM

MODEL = "gpt-4o-mini"  # selectable: e.g. "gpt-4o", "claude-3-haiku-20240307"
TOP_K = 10             # number of web search results to retrieve

## Step 1 — Web Search

`search_web` calls the Tavily API and returns the top K results.
Each result contains a **title**, **URL**, and a **content snippet**.

In [ ]:
from tavily import TavilyClient

def search_web(query, k=TOP_K):
    """Search the web via Tavily; return top-k results."""
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(query, max_results=k)
    return response["results"]  # each item: {title, url, content, score, ...}

## Step 2 — LLM Generation

`generate_answer` builds a prompt with the query and all search results, then calls
the LLM via LiteLLM. The LLM returns markdown with two sections:
- `## Answer` — synthesized answer with inline `[Title](URL)` citations
- `## Web Search Results` — all results listed with clickable titles and snippets

In [ ]:
import litellm
litellm.set_verbose = False  # suppress debug output

def generate_answer(query, results, model=MODEL):
    """Send query + search results to LLM; return markdown-formatted answer."""
    # Format each result for the prompt
    results_text = "\n\n".join(
        f"[{i+1}] Title: {r['title']}\nURL: {r['url']}\nSnippet: {r['content']}"
        for i, r in enumerate(results)
    )
    n = len(results)

    system_prompt = (
        "You are a helpful research assistant. "
        "Answer the user's query using ONLY the provided search results. "
        "Do not consider results that are irrelevant to the query. "
        "Add inline citations as markdown links [Title](URL) near supporting text. "
        f"End with a '## Web Search Results' section listing ALL {n} results as:\n"
        "### [Title](URL)\nSnippet text"
    )
    user_prompt = f"Question: {query}\n\nSearch Results:\n{results_text}"

    response = litellm.completion(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
    )
    return response.choices[0].message.content  # markdown string

## Step 3 — Display Answer

`display_answer` converts the LLM's markdown output to HTML and renders it inline.
This makes citations clickable and the answer easy to read inside Colab.

In [ ]:
import markdown
from IPython.display import display, HTML

def display_answer(answer_md):
    """Convert markdown answer to HTML and render it in the notebook."""
    html = markdown.markdown(answer_md, extensions=["extra"])
    display(HTML(html))

## Run

Edit the `query` below and run this cell to search and display the answer.

In [ ]:
query = "What are the latest AI breakthroughs in 2025?"  # <- change this

results = search_web(query)
answer_md = generate_answer(query, results)

display_answer(answer_md)